# Notebook 4: Delinquency & Cohort Analysis (Advanced)
Multi-step workflow combining SQL CTEs, Python analytics, Jinja cross-referencing, and visualization.

### Payment Performance by Loan (CTE)
This query uses a Common Table Expression (CTE) to first aggregate payment-level data per loan — counting on-time, partial, and missed payments, and calculating total amounts due vs paid. It then joins this payment summary with loan and customer data to produce a complete picture of each loan's payment performance, including collection rate and average days late.

In [ ]:
%%sql -r delinquency_data
WITH payment_stats AS (
    SELECT LOAN_ID,
           COUNT(*) AS TOTAL_PAYMENTS,
           SUM(CASE WHEN PAYMENT_STATUS = 'ON_TIME' THEN 1 ELSE 0 END) AS ON_TIME_COUNT,
           SUM(CASE WHEN PAYMENT_STATUS = 'PARTIAL' THEN 1 ELSE 0 END) AS PARTIAL_COUNT,
           SUM(CASE WHEN PAYMENT_STATUS = 'MISSED' THEN 1 ELSE 0 END) AS MISSED_COUNT,
           SUM(AMOUNT_DUE) AS TOTAL_DUE,
           SUM(AMOUNT_PAID) AS TOTAL_PAID,
           MAX(DAYS_LATE) AS MAX_DAYS_LATE,
           AVG(DAYS_LATE) AS AVG_DAYS_LATE
    FROM LOAN_DB.LENDING.PAYMENTS
    GROUP BY LOAN_ID
)
SELECT l.LOAN_ID, l.LOAN_TYPE, l.LOAN_AMOUNT, l.INTEREST_RATE,
       l.LOAN_STATUS, l.ORIGINATION_DATE, l.TERM_MONTHS,
       c.CREDIT_SCORE, c.ANNUAL_INCOME, c.EMPLOYMENT_STATUS, c.STATE,
       ps.TOTAL_PAYMENTS, ps.ON_TIME_COUNT, ps.PARTIAL_COUNT, ps.MISSED_COUNT,
       ps.TOTAL_DUE, ps.TOTAL_PAID, ps.MAX_DAYS_LATE,
       ROUND(ps.AVG_DAYS_LATE, 1) AS AVG_DAYS_LATE,
       ROUND((ps.TOTAL_PAID / NULLIF(ps.TOTAL_DUE, 0)) * 100, 1) AS COLLECTION_RATE
FROM LOAN_DB.LENDING.LOANS l
JOIN LOAN_DB.LENDING.CUSTOMERS c ON l.CUSTOMER_ID = c.CUSTOMER_ID
LEFT JOIN payment_stats ps ON l.LOAN_ID = ps.LOAN_ID
ORDER BY ps.MISSED_COUNT DESC NULLS LAST

### Classify Loans into Risk Buckets
This cell applies a custom risk classification to each loan based on its payment behaviour. Loans with 2 or more missed payments or 60+ days late are flagged as HIGH risk; those with 1 missed payment, 2+ partial payments, or 15+ days late are MEDIUM; all others are LOW. A summary table then aggregates loan count, total exposure, average collection rate, and average max days late per risk bucket.

In [ ]:
import pandas as pd
import numpy as np

df = delinquency_data.copy()

def classify_risk(row):
    if row['MISSED_COUNT'] >= 2 or row['MAX_DAYS_LATE'] >= 60:
        return 'HIGH'
    elif row['MISSED_COUNT'] >= 1 or row['PARTIAL_COUNT'] >= 2 or row['MAX_DAYS_LATE'] >= 15:
        return 'MEDIUM'
    else:
        return 'LOW'

df['RISK_BUCKET'] = df.apply(classify_risk, axis=1)

risk_summary = df.groupby('RISK_BUCKET').agg(
    loan_count=('LOAN_ID', 'count'),
    total_exposure=('LOAN_AMOUNT', 'sum'),
    avg_collection_rate=('COLLECTION_RATE', 'mean'),
    avg_max_days_late=('MAX_DAYS_LATE', 'mean')
).round(2)
risk_summary

### Origination Cohort Analysis
This cell groups loans by the quarter they were originated and tracks how each cohort has performed over time. For each quarter it shows the number of loans, total amount, average collection rate, average credit score, and the percentage of loans that have fallen into the HIGH risk bucket. This helps identify whether newer or older cohorts are performing better or worse.

In [ ]:
df['ORIGINATION_DATE'] = pd.to_datetime(df['ORIGINATION_DATE'])
df['ORIGINATION_QUARTER'] = df['ORIGINATION_DATE'].dt.to_period('Q').astype(str)

cohort = df.groupby('ORIGINATION_QUARTER').agg(
    loan_count=('LOAN_ID', 'count'),
    total_amount=('LOAN_AMOUNT', 'sum'),
    avg_collection_rate=('COLLECTION_RATE', 'mean'),
    high_risk_count=('RISK_BUCKET', lambda x: (x == 'HIGH').sum()),
    avg_credit_score=('CREDIT_SCORE', 'mean')
).round(2)
cohort['high_risk_pct'] = (cohort['high_risk_count'] / cohort['loan_count'] * 100).round(1)
cohort

### Cohort & Risk Visualizations
This cell produces a 2x2 grid of charts summarising the risk and cohort analysis. The top row shows loan count and dollar exposure split by risk bucket (LOW / MEDIUM / HIGH). The bottom row shows how average collection rate and high-risk percentage have trended across origination quarters, making it easy to spot deteriorating or improving cohorts over time.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

risk_colors = {'LOW': '#2ecc71', 'MEDIUM': '#f39c12', 'HIGH': '#e74c3c'}
ordered_risk = ['LOW', 'MEDIUM', 'HIGH']
counts = [risk_summary.loc[r, 'loan_count'] if r in risk_summary.index else 0 for r in ordered_risk]
axes[0,0].bar(ordered_risk, counts, color=[risk_colors[r] for r in ordered_risk])
axes[0,0].set_title('Loan Count by Risk Bucket')
axes[0,0].set_ylabel('Count')

exposure = [risk_summary.loc[r, 'total_exposure'] if r in risk_summary.index else 0 for r in ordered_risk]
axes[0,1].pie(exposure, labels=ordered_risk, colors=[risk_colors[r] for r in ordered_risk],
              autopct='%1.1f%%', startangle=90)
axes[0,1].set_title('Exposure by Risk Bucket')

cohort_plot = cohort.reset_index()
axes[1,0].plot(cohort_plot['ORIGINATION_QUARTER'], cohort_plot['avg_collection_rate'],
               marker='o', color='#3498db', linewidth=2)
axes[1,0].set_title('Collection Rate by Cohort')
axes[1,0].set_ylabel('Avg Collection Rate (%)')
axes[1,0].tick_params(axis='x', rotation=45)

axes[1,1].bar(cohort_plot['ORIGINATION_QUARTER'], cohort_plot['high_risk_pct'],
              color='#e74c3c', alpha=0.7)
axes[1,1].set_title('High Risk % by Cohort')
axes[1,1].set_ylabel('High Risk (%)')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### Prepare High Risk IDs for SQL Drill-down
This cell extracts the loan IDs of all HIGH risk loans from the classified dataset and formats them as a comma-separated string. This list is then ready to be passed into the next SQL query as a filter, enabling a targeted drill-down into the payment history of only the most at-risk loans.

In [ ]:
high_risk_threshold = 'HIGH'
high_risk_ids = df[df['RISK_BUCKET'] == high_risk_threshold]['LOAN_ID'].tolist()
high_risk_ids_str = ','.join(str(x) for x in high_risk_ids)
print(f"High risk loan IDs for drill-down: {high_risk_ids_str}")

### Troubled Loans - Payment Timeline
This query retrieves the full payment history for loans currently in DELINQUENT or DEFAULT status. It uses a window function to calculate a running cumulative total of payments made per loan over time. The results are ordered by loan and payment date, giving a chronological view of how each troubled loan's repayment has progressed.

In [ ]:
%%sql -r troubled_payment_history
SELECT p.LOAN_ID, l.LOAN_TYPE, l.LOAN_STATUS,
       p.PAYMENT_DATE, p.AMOUNT_DUE, p.AMOUNT_PAID,
       p.PAYMENT_STATUS, p.DAYS_LATE,
       SUM(p.AMOUNT_PAID) OVER (PARTITION BY p.LOAN_ID ORDER BY p.PAYMENT_DATE) AS CUMULATIVE_PAID
FROM LOAN_DB.LENDING.PAYMENTS p
JOIN LOAN_DB.LENDING.LOANS l ON p.LOAN_ID = l.LOAN_ID
WHERE l.LOAN_STATUS IN ('DELINQUENT', 'DEFAULT')
ORDER BY p.LOAN_ID, p.PAYMENT_DATE

### Troubled Loan Payment Trajectories
This cell plots the cumulative payment trajectory for each troubled loan on a single chart, with one line per loan. It allows you to visually compare how quickly different troubled loans have been paying down and whether any show a sudden drop-off or plateau in payments — which could indicate an impending default or an already deteriorating situation.

In [ ]:
tph = troubled_payment_history.copy()
tph['PAYMENT_DATE'] = pd.to_datetime(tph['PAYMENT_DATE'])

fig, ax = plt.subplots(figsize=(14, 6))
for loan_id in tph['LOAN_ID'].unique():
    subset = tph[tph['LOAN_ID'] == loan_id]
    label = f"Loan {loan_id} ({subset['LOAN_STATUS'].iloc[0]})"
    ax.plot(subset['PAYMENT_DATE'], subset['CUMULATIVE_PAID'], marker='o', label=label)

ax.set_title('Cumulative Payments for Troubled Loans')
ax.set_xlabel('Payment Date')
ax.set_ylabel('Cumulative Amount Paid ($)')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()